# Gallery

> Main gallery component that combines controls, views, and preview.

In [ ]:
#| default_exp components.gallery

In [ ]:
#| export
from typing import Any, Callable, List, Optional

from fasthtml.common import Div

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, flex_direction, flex
from cjm_fasthtml_tailwind.utilities.borders import rounded, border
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.core.base import combine_classes

# DaisyUI utilities
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, border_dui

# Local imports
from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_fasthtml_media_gallery.core.config import GalleryConfig, ViewMode
from cjm_fasthtml_media_gallery.core.html_ids import GalleryHtmlIds
from cjm_fasthtml_media_gallery.components.controls import render_gallery_controls
from cjm_fasthtml_media_gallery.components.grid_view import (
    render_grid_view, render_grid_empty_state
)
from cjm_fasthtml_media_gallery.components.list_view import (
    render_list_view, render_list_empty_state
)
from cjm_fasthtml_media_gallery.components.preview import render_preview_modal
from cjm_fasthtml_media_gallery.patterns.pagination import render_pagination, PaginationInfo

## Gallery Content

The inner content area that switches between grid and list views.

In [ ]:
#| export
def render_gallery_content(
    files: List[FileInfo],            # Files to display
    config: GalleryConfig,            # Gallery configuration
    view_mode: ViewMode,              # Current view mode
    get_file_url: Optional[Callable[[str], str]] = None,  # Function to get file URL
    selected_paths: Optional[List[str]] = None,  # Currently selected paths
    preview_url: Optional[str] = None,  # URL for preview action
    select_url: Optional[str] = None,   # URL for selection action
    hx_target: Optional[str] = None,    # HTMX target for swaps
    start_index: int = 0,               # Starting index (for pagination)
) -> Any:  # Content component
    """Render the gallery content (grid or list view)."""
    # Handle empty state
    if not files:
        if view_mode == ViewMode.GRID:
            return render_grid_empty_state()
        else:
            return render_list_empty_state()
    
    # Render appropriate view
    if view_mode == ViewMode.GRID:
        return render_grid_view(
            files=files,
            config=config,
            get_file_url=get_file_url,
            selected_paths=selected_paths,
            preview_url=preview_url,
            select_url=select_url,
            hx_target=hx_target,
            start_index=start_index
        )
    else:
        return render_list_view(
            files=files,
            config=config,
            selected_paths=selected_paths,
            preview_url=preview_url,
            select_url=select_url,
            hx_target=hx_target,
            start_index=start_index
        )

In [ ]:
from fasthtml.common import to_xml

# Test gallery content - grid view
config = GalleryConfig()
files = [
    FileInfo(name="photo.jpg", path="/photo.jpg", is_directory=False, file_type=FileType.IMAGE),
    FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, file_type=FileType.AUDIO),
]

content = render_gallery_content(
    files=files,
    config=config,
    view_mode=ViewMode.GRID,
    preview_url="/preview"
)
html = to_xml(content)
assert 'id="media-gallery-grid"' in html

# Test list view
content = render_gallery_content(
    files=files,
    config=config,
    view_mode=ViewMode.LIST,
    preview_url="/preview"
)
html = to_xml(content)
assert 'id="media-gallery-list"' in html

# Test empty state
content = render_gallery_content(
    files=[],
    config=config,
    view_mode=ViewMode.GRID
)
html = to_xml(content)
assert "No media files found" in html

print("render_gallery_content tests passed!")

render_gallery_content tests passed!


## Main Gallery Component

The complete gallery with controls, content, and preview modal.

In [ ]:
#| export
def render_media_gallery(
    files: List[FileInfo],            # Files to display
    config: GalleryConfig,            # Gallery configuration
    view_mode: Optional[ViewMode] = None,  # Current view mode (defaults to config)
    active_types: Optional[List[FileType]] = None,  # Active type filters
    get_file_url: Optional[Callable[[str], str]] = None,  # Function to get file URL
    selected_paths: Optional[List[str]] = None,  # Currently selected paths
    # Route URLs
    toggle_view_url: Optional[str] = None,  # URL for view toggle
    filter_url: Optional[str] = None,       # URL for type filter
    preview_url: Optional[str] = None,      # URL for preview action
    select_url: Optional[str] = None,       # URL for selection action
    page_url: Optional[str] = None,         # URL for pagination
    # Pagination
    current_page: int = 1,                  # Current page number
    total_items: Optional[int] = None,      # Total items (for pagination)
    type_counts: Optional[dict[FileType, int]] = None,  # File counts per type
) -> Any:  # Complete gallery component
    """Render the complete media gallery."""
    # Defaults
    view_mode = view_mode or config.default_view
    active_types = active_types if active_types is not None else config.filter.enabled_types
    selected_paths = selected_paths or []
    
    # HTMX target for content swaps
    content_target = GalleryHtmlIds.as_selector(config.content_id)
    gallery_target = GalleryHtmlIds.as_selector(config.gallery_id)
    
    # Filter files by active types
    filtered_files = [
        f for f in files
        if f.file_type in active_types and config.filter.matches(f)
    ]
    
    # Calculate pagination info
    total_filtered = len(filtered_files)
    page_size = config.pagination.items_per_page
    start_idx = (current_page - 1) * page_size
    end_idx = start_idx + page_size
    paginated_files = filtered_files[start_idx:end_idx]
    
    # Build gallery
    components = []
    
    # Controls bar
    if config.allow_view_toggle or config.filter.allow_type_filter:
        components.append(
            render_gallery_controls(
                config=config,
                current_view=view_mode,
                active_types=active_types,
                toggle_view_url=toggle_view_url or "",
                filter_url=filter_url or "",
                hx_target=gallery_target,
                type_counts=type_counts,
                controls_id=config.controls_id
            )
        )
    
    # Content area
    content = render_gallery_content(
        files=paginated_files,
        config=config,
        view_mode=view_mode,
        get_file_url=get_file_url,
        selected_paths=selected_paths,
        preview_url=preview_url,
        select_url=select_url,
        hx_target=gallery_target,
        start_index=start_idx
    )
    
    components.append(
        Div(
            content,
            id=config.content_id,
            cls=combine_classes(bg_dui.base_100, overflow.auto, flex(1))
        )
    )
    
    # Pagination controls
    if config.pagination.show_pagination and page_url:
        pagination_info = PaginationInfo(
            total_items=total_filtered,
            items_per_page=page_size,
            current_page=current_page
        )
        pagination_component = render_pagination(
            info=pagination_info,
            page_url=page_url,
            hx_target=gallery_target
        )
        if pagination_component:  # None if only 1 page
            components.append(pagination_component)
    
    # Preview modal
    if config.preview.enable_preview:
        components.append(
            render_preview_modal(modal_id=config.preview_modal_id)
        )
    
    # Container
    container_cls = combine_classes(
        flex_display, flex_direction.col,
        w.full, h.full,
        border(), border_dui.base_300, rounded.lg,
        overflow.hidden
    )
    
    return Div(
        *components,
        id=config.gallery_id,
        cls=container_cls
    )

In [ ]:
# Test main gallery
config = GalleryConfig()
files = [
    FileInfo(name="photo1.jpg", path="/photo1.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg"),
    FileInfo(name="photo2.jpg", path="/photo2.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg"),
    FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, file_type=FileType.AUDIO, extension="mp3"),
    FileInfo(name="video.mp4", path="/video.mp4", is_directory=False, file_type=FileType.VIDEO, extension="mp4"),
]

gallery = render_media_gallery(
    files=files,
    config=config,
    toggle_view_url="/toggle-view",
    filter_url="/filter",
    preview_url="/preview",
    page_url="/page"
)
html = to_xml(gallery)

# Check main container
assert 'id="media-gallery"' in html

# Check controls
assert 'id="media-gallery-controls"' in html
assert "join" in html  # View toggle
assert "Audio" in html  # Type filters

# Check content
assert 'id="media-gallery-content"' in html
assert 'id="media-gallery-grid"' in html  # Default is grid

# Check preview modal
assert 'id="media-preview-modal"' in html

# Check files are rendered
assert "photo1.jpg" in html
assert "song.mp3" in html

print("render_media_gallery tests passed!")

render_media_gallery tests passed!


In [ ]:
# Test with list view
config = GalleryConfig()
files = [
    FileInfo(name="photo1.jpg", path="/photo1.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg"),
    FileInfo(name="photo2.jpg", path="/photo2.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg"),
    FileInfo(name="song.mp3", path="/song.mp3", is_directory=False, file_type=FileType.AUDIO, extension="mp3"),
    FileInfo(name="video.mp4", path="/video.mp4", is_directory=False, file_type=FileType.VIDEO, extension="mp4"),
]

gallery = render_media_gallery(
    files=files,
    config=config,
    view_mode=ViewMode.LIST
)
html = to_xml(gallery)
assert 'id="media-gallery-list"' in html
assert "<table" in html

# Test with type filter
gallery = render_media_gallery(
    files=files,
    config=config,
    active_types=[FileType.IMAGE]  # Only images
)
html = to_xml(gallery)
assert "photo1.jpg" in html
assert "photo2.jpg" in html
assert "song.mp3" not in html  # Audio filtered out
assert "video.mp4" not in html  # Video filtered out

# Test with preview disabled
from cjm_fasthtml_media_gallery.core.config import PreviewConfig
no_preview_config = GalleryConfig(preview=PreviewConfig(enable_preview=False))
gallery = render_media_gallery(
    files=files,
    config=no_preview_config
)
html = to_xml(gallery)
assert 'id="media-preview-modal"' not in html

print("Additional gallery tests passed!")

Additional gallery tests passed!


In [ ]:
# Test pagination controls
from cjm_fasthtml_media_gallery.core.config import PaginationConfig

# Create many files to trigger pagination
many_files = [
    FileInfo(name=f"file_{i}.jpg", path=f"/file_{i}.jpg", is_directory=False, file_type=FileType.IMAGE, extension="jpg")
    for i in range(50)
]

# Config with small page size to trigger pagination
paginated_config = GalleryConfig(
    pagination=PaginationConfig(items_per_page=10, show_pagination=True)
)

gallery = render_media_gallery(
    files=many_files,
    config=paginated_config,
    page_url="/page",
    current_page=1
)
html = to_xml(gallery)

# Check pagination is rendered
assert 'id="media-gallery-pagination"' in html
assert "Next" in html or "next" in html.lower()  # Next button

# Check only 10 items shown (page 1)
assert "file_0.jpg" in html
assert "file_9.jpg" in html
assert "file_10.jpg" not in html  # Should be on page 2

# Test page 2
gallery = render_media_gallery(
    files=many_files,
    config=paginated_config,
    page_url="/page",
    current_page=2
)
html = to_xml(gallery)
assert "file_10.jpg" in html
assert "file_19.jpg" in html
assert "file_0.jpg" not in html  # Should be on page 1

# Test without page_url (no pagination rendered)
gallery = render_media_gallery(
    files=many_files,
    config=paginated_config,
    page_url=None  # No page URL
)
html = to_xml(gallery)
assert 'id="media-gallery-pagination"' not in html

print("Pagination tests passed!")

Pagination tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()